## Near-Duplicate Image Detection using Perceptual Hashing (pHash)

In [ ]:
!pip install ImageHash

In [2]:
import os
from pathlib import Path
from collections import defaultdict

from PIL import Image
import imagehash


In [17]:
DATASET_ROOT = Path("../data")

splits = ["train", "validation", "test"]

In [18]:
Hash_distance_threshold = 5  

In [19]:
Image_extensions = {
    ".jpg", 
    ".jpeg", 
    ".png", 
    ".webp"
}

In [34]:
def collect_image_paths():
    image_paths = []

    for split in splits:
        split_path = DATASET_ROOT / split

        if not split_path.exists():
            print(f"WARNING: Directory not found: {split_path}")
            continue

        for class_dir in split_path.iterdir():

            if not class_dir.is_dir():
                continue

            class_name = class_dir.name

            for image_path in class_dir.iterdir():

                if image_path.is_file() and image_path.suffix.lower() in Image_extensions:
                    image_paths.append((
                    image_path,
                    split,
                    class_name,
                ))
    return image_paths
                

###  CALCULATE PERCEPTUAL HASH

In [35]:
def calculate_phash(image_path):
    try:
        with Image.open(image_path) as img:
                       img = img.convert("RGB")

                       image_hash = imagehash.phash(img)

        return image_hash

    except Exception as error:
        print(f"Could not process: {image_path}")
        print(f"Error: {error}")

        return None

In [37]:
def calculate_all_hashes(image_paths):
    hashed_images = []

    print("\nCalculating perceptual hashes...")
    print("-" * 60)

    for index, (image_path, split, class_name) in enumerate(image_paths, start = 1):
        image_hash = calculate_phash(image_path)

        if image_hash is not None:
            hashed_images.append((
                image_path,
                split,
                class_name,
                image_hash,
            ))

        # shows progress every 500 imges
        if index % 500 == 0:
            print(f"processed {index}/{len(image_paths)} images")

        print(f"Finished processing {len(hashed_images)} images.")

    return hashed_images
    

### Find Near-Duplicates

In [38]:
def find_near_duplicates(hashed_images):
    near_duplicates = []
    total_images = len(hashed_images)

    print("\nSearching for near duplicated...")
    print("-" * 60)

    for i in range(total_images):
        image_path_1, split_1, class_1, hash_1 = hashed_images[i]

        for j in range(i + 1, total_images):
            image_path_2, split_2, class_2, hash_2 = hashed_images[j]

            distance = hash_1 - hash_2

            if distance <= Hash_distance_threshold:
                near_duplicates.append({
                    "image1": image_path_1,
                    "split1": split_1,
                    "class1": class_1,

                    "image2": image_path_2,
                    "split2": split_2,
                    "class2": class_2,

                    "distance": distance,
                })
        if (i + 1) % 500 == 0:
            print(f"processed {i + 1}/{total_images} images")

    return near_duplicates

In [30]:
def print_results(near_duplicates):
    print(f"\n")
    print("=" * 70)
    print("Near-Duplicate Detection Results")
    print("=" * 70)

    print(f"Hash distance threshold: {Hash_distance_threshold}")
    print(f"Near-Duplicate pairs found: {len(near_duplicates)}")

    if not near_duplicates:
        print("\nNo near-duplicate pairs found.")
        return

    cross_split_pairs = [
        pair
        for pair in near_duplicates
        if pair["split1"] != pair["split2"]
    ]

    print(f"Cross-split near-duplicate pairs: {len(cross_split_pairs)}")

    cross_class_pairs = [
    pair
    for pair in near_duplicates
    if pair["class1"] != pair["class2"]
    ]

    print(f"Cross-class near-duplicate pairs: {len(cross_class_pairs)}")

    print("\n")
    print("=" * 70)
    print("Near-Duplicate Pairs")
    print("=" * 70)

    for number, pair in enumerate(near_duplicates, start = 1):

        print(f"\nPair {number}")
        print(f"Distance : {pair['distance']}")

        print(
            f"Image 1  : {pair['image1']}"
        )

        print(
            f"          Split = {pair['split1']}, "
            f"Class = {pair['class1']}"
        )

        print(
            f"Image 2  : {pair['image2']}"
        )

        print(
            f"          Split = {pair['split2']}, "
            f"Class = {pair['class2']}"
        )

        # HIghlight potentially important cases
        if pair["split1"] != pair["split2"]:
            print("CROSS-SPLIT: Needs manual inspection!")

        if pair["class1"] != pair["class2"]:
            print("CROSS-CLASS: Possible labeling issue!")

In [39]:
# saves results to a text file

def save_results(near_duplicates):
    #Create a directory to store the results if it doesn't exist
    results_dir = Path("../results") 
    results_dir.mkdir(exist_ok=True)

    output_file = results_dir / "near_duplicate_results.txt"

    with open(output_file, "w", encoding="utf-8") as file:
        file.write("NEAR-DUPLICATE IMAGE DETECTION RESULTS\n")
        file.write("=" * 70 + "\n\n")

        file.write(f"Hash distance threshold:" f"{Hash_distance_threshold}\n")

        file.write(f"Near-duplicate pairs:" f"{len(near_duplicates)}\n\n")

        for number, pair in enumerate(near_duplicates, start=1):
            file.write(f"Pair {number}\n")
            file.write("=" * 50 + "\n")

            file.write(f"Distance: {pair['distance']}\n")
            file.write(f"Image 1: {pair['image1']}\n")
            file.write(f"Split 1: {pair['split1']}\n")
            file.write(f"Class 1: {pair['class1']}\n\n")

            file.write(f"Image 2: {pair['image2']}\n")
            file.write(f"Split 2: {pair['split2']}\n")
            file.write(f"class 2: {pair['class2']}\n\n")

            if pair["split1"] != pair["split2"]:
                file.write("Warning: Cross-Split near duplicate\n")
            file.write("\n")
    print(f"\nResults saved to: {output_file}")

In [40]:
def main():
    print("=" * 70)
    print("DOMESTIC ANIMAL DATASET - NEAR DUPLICATE DETECTION")

    image_paths = collect_image_paths()

    print(f"\nTotal images found: {len(image_paths)}")

    hashed_images = calculate_all_hashes(image_paths)

    near_duplicates = find_near_duplicates(hashed_images)

    print_results(near_duplicates)

    save_results(near_duplicates)

    print("\n")
    print("=" * 70)
    print("NEAR DUPLICATE DETECTION COMPLETED")
    print("=" * 70)

In [41]:
if __name__ == "__main__":
    main()

DOMESTIC ANIMAL DATASET - NEAR DUPLICATE DETECTION

Total images found: 10760

Calculating perceptual hashes...
------------------------------------------------------------
Finished processing 1 images.
Finished processing 2 images.
Finished processing 3 images.
Finished processing 4 images.
Finished processing 5 images.
Finished processing 6 images.
Finished processing 7 images.
Finished processing 8 images.
Finished processing 9 images.
Finished processing 10 images.
Finished processing 11 images.
Finished processing 12 images.
Finished processing 13 images.
Finished processing 14 images.
Finished processing 15 images.
Finished processing 16 images.
Finished processing 17 images.
Finished processing 18 images.
Finished processing 19 images.
Finished processing 20 images.
Finished processing 21 images.
Finished processing 22 images.
Finished processing 23 images.
Finished processing 24 images.
Finished processing 25 images.
Finished processing 26 images.
Finished processing 27 images.

C:\Users\Tushar\AppData\Roaming\Python\Python313\site-packages\PIL\Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Finished processing 7040 images.
Finished processing 7041 images.
Finished processing 7042 images.
Finished processing 7043 images.
Finished processing 7044 images.
Finished processing 7045 images.
Finished processing 7046 images.
Finished processing 7047 images.
Finished processing 7048 images.
Finished processing 7049 images.
Finished processing 7050 images.
Finished processing 7051 images.
Finished processing 7052 images.
Finished processing 7053 images.
Finished processing 7054 images.
Finished processing 7055 images.
Finished processing 7056 images.
Finished processing 7057 images.
Finished processing 7058 images.
Finished processing 7059 images.
Finished processing 7060 images.
Finished processing 7061 images.
Finished processing 7062 images.
Finished processing 7063 images.
Finished processing 7064 images.
Finished processing 7065 images.
Finished processing 7066 images.
Finished processing 7067 images.
Finished processing 7068 images.
Finished processing 7069 images.
Finished p